# 25 — Veteran Homeless by CoC

Examines veteran homelessness — veteran rate per 10k veterans vs overall homeless rate,
and which CoCs have disproportionately high veteran homeless rates.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_city_data.csv')
df = df.dropna(subset=['homeless_rate_per_10k', 'veteran_rate_per_10k', 'veteran_homeless'])
df = df[df['veteran_homeless'] > 0]
print(f'Loaded {len(df)} CoCs with veteran data')

Loaded 102 CoCs with veteran data


In [2]:
slope, intercept, r, p, se = stats.linregress(df['homeless_rate_per_10k'], df['veteran_rate_per_10k'])
r2 = r ** 2
print(f'Overall vs veteran homeless rate: r={r:.3f}, R²={r2:.3f}, p={p:.4f}')

x_range = [df['homeless_rate_per_10k'].min(), df['homeless_rate_per_10k'].max()]
y_range = [slope * x + intercept for x in x_range]

fig1 = px.scatter(
    df, x='homeless_rate_per_10k', y='veteran_rate_per_10k',
    size='veteran_homeless',
    color='veteran_rate_per_10k', color_continuous_scale='Blues',
    text='city_name',
    hover_name='coc_name',
    hover_data={'state_postal': True, 'veteran_homeless': ':,', 'homeless_rate_per_10k': ':.1f', 'veteran_rate_per_10k': ':.1f'},
    title=f'Overall vs. Veteran Homeless Rate by CoC<br><sup>r={r:.3f}, R²={r2:.3f}, p={p:.4f} — bubble size = veteran homeless count</sup>',
    labels={'homeless_rate_per_10k': 'Overall Homeless Rate per 10k', 'veteran_rate_per_10k': 'Veteran Homeless Rate per 10k Veterans'},
    size_max=50,
)
fig1.add_trace(go.Scatter(x=x_range, y=y_range, mode='lines', name='Regression', line=dict(color='navy', dash='dash')))
fig1.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig1.show()

Overall vs veteran homeless rate: r=0.975, R²=0.950, p=0.0000


In [3]:
top20 = df.nlargest(20, 'veteran_homeless')
fig2 = px.bar(
    top20.sort_values('veteran_homeless', ascending=True),
    x='veteran_homeless', y='city_name', orientation='h',
    color='veteran_rate_per_10k', color_continuous_scale='Blues',
    title='Top 20 CoCs by Veteran Homeless Count (HUD 2023)',
    labels={'veteran_homeless': 'Veteran Homeless Count', 'city_name': ''},
)
fig2.show()

In [4]:
df['veteran_share_pct'] = (df['veteran_homeless'] / df['total_homeless'] * 100).round(1)
print(f'National veteran share of homeless: {df["veteran_share_pct"].mean():.1f}%')
cols = ['city_name', 'state_postal', 'veteran_share_pct', 'veteran_homeless']
print('Highest veteran share:')
print(df.nlargest(5, 'veteran_share_pct')[cols].to_string(index=False))

National veteran share of homeless: 9.1%
Highest veteran share:
       city_name state_postal  veteran_share_pct  veteran_homeless
          Austin           TX               11.5               212
        Portland           OR               11.3               712
Colorado Springs           CO               11.2               172
      Fort Worth           TX               10.9               172
      Sacramento           CA               10.8               312
